# 🧬 反应堆物理数值计算：从中子扩散方程到点堆动力学

> 5 分钟跑通全链路：FDM 离散 → 幂迭代 → 临界搜索 → 2D 扩展 → 瞬态动力学

**运行环境**: Python 3.10+, NumPy, SciPy, Matplotlib  
**GitHub**: [neutron-diffusion-solver](https://github.com/buzhidao2006/neutron-diffusion-solver)


## 1. 物理背景：中子在哪里？

核反应堆的本质是一个**中子增殖系统**。控制棒、硼酸、燃料装载，全部围绕一个核心问题：

$$k_{\text{eff}} = \frac{\text{下一代裂变中子数}}{\text{当前代裂变中子数}}$$

- $k > 1$：超临界 → 功率上升
- $k = 1$：临界 → 稳态运行  
- $k < 1$：次临界 → 停堆

中子输运的精确方程是玻尔兹曼输运方程，但直接用 Fick 定律近似，得到**中子扩散方程**：

$$-D\nabla^2\phi(\mathbf{r}) + \Sigma_a\phi(\mathbf{r}) = \frac{1}{k}\nu\Sigma_f\phi(\mathbf{r})$$

| 项 | 物理含义 |
|---|---|
| $-D\nabla^2\phi$ | 泄漏 — 中子从高浓度区域扩散出去 |
| $\Sigma_a\phi$ | 吸收 — 中子被材料吸收 |
| $\frac{1}{k}\nu\Sigma_f\phi$ | 裂变源 — 裂变产生新中子（需要除以 $k$ 平衡） |

这本质上是一个**特征值问题**：求 $k$ 使方程有非零解。$k$ 是特征值，$\phi$ 是特征向量（通量形状）。


## 2. 有限差分法：把连续方程变成矩阵

把一维平板 $[0, L]$ 切成 $N$ 份，每份宽 $h = L/N$。二阶导数用**中心差分**近似：

$$\frac{d^2\phi}{dx^2}\bigg|_{x_i} \approx \frac{\phi_{i-1} - 2\phi_i + \phi_{i+1}}{h^2}$$

代入扩散方程，整理成矩阵形式 $A\phi = \frac{1}{k}F\phi$，其中 $A$ 是三对角矩阵：

$$A = \frac{D}{h^2}\begin{bmatrix} 2 & -1 & 0 & \cdots \\ -1 & 2 & -1 & \cdots \\ 0 & -1 & 2 & \cdots \\ \vdots & \vdots & \vdots & \ddots \end{bmatrix} + \Sigma_a I$$


In [ ]:
import numpy as np
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt

# ===== 截面数据 (典型 PWR) =====
D = 1.2          # 扩散系数 (cm)
nu_Sf = 0.020    # nu * Sigma_f (cm^-1)
Sa = 0.015       # 吸收截面 (cm^-1)
L = 200.0        # 平板半厚度 (cm)
N = 150          # 网格点数


In [ ]:
# ===== 构造矩阵 A =====
h = L / N
coeff = D / (h * h)

# 三对角矩阵: [2, -1, -1, 2, -1, ...]
diag = 2 * np.ones(N)
off_diag = -np.ones(N - 1)
L_mat = diags([off_diag, diag, off_diag], [-1, 0, 1])

A = coeff * L_mat + Sa * diags([np.ones(N)], [0])  # A = coeff*L + Sa*I
F = nu_Sf * diags([np.ones(N)], [0])               # F = nu_Sf * I

print(f'矩阵 A 尺寸: {A.shape}, 类型: {type(A).__name__}')
print(f'矩阵 F 尺寸: {F.shape}')


In [ ]:
# ===== 幂迭代求解 k_eff 和通量 =====
phi = np.ones(N)        # 初始猜测
phi = phi / np.max(phi)
k_eff = 1.0

k_history = []
for iteration in range(200):
    source = F @ phi                      # 当前裂变源
    phi_new = spsolve(A, source)           # 解 A·phi = source
    source_new = F @ phi_new               # 新裂变源
    k_new = np.sum(source_new) / np.sum(source)  # Rayleigh 商
    
    k_history.append(k_new)
    
    if abs(k_new - k_eff) < 1e-10:
        break
    
    k_eff = k_new
    phi = phi_new / np.max(phi_new)        # 归一化

x = np.linspace(0, L, N) + h/2
print(f'k_eff = {k_eff:.6f}  (迭代 {len(k_history)} 次)')
status = "超临界" if k_eff > 1.001 else ("次临界" if k_eff < 0.999 else "临界")
print(f'状态: {status}')


In [ ]:
# ===== 可视化 =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 通量分布
ax1.plot(x, phi, 'b-', linewidth=2)
ax1.fill_between(x, 0, phi, alpha=0.1, color='b')
ax1.set_xlabel('位置 (cm)')
ax1.set_ylabel('中子通量 (归一化)')
ax1.set_title(f'一维单群中子通量分布  |  k_eff = {k_eff:.6f}')
ax1.grid(True, alpha=0.3)

# 收敛历史
ax2.semilogy(range(len(k_history)), np.abs(np.array(k_history) - k_eff), 'r.-', markersize=4)
ax2.set_xlabel('迭代次数')
ax2.set_ylabel('|k - k_final|')
ax2.set_title('幂迭代收敛历史')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Debug 方法论：解析特征值验证

**"代码跑了不等于对"** — 这是数值计算的第一课。我们需要用解析公式验证矩阵构造。

三对角矩阵 $(2, -1, -1)$ 的特征值**解析已知**：

$$\lambda_k = 4\sin^2\left(\frac{k\pi}{2(N+1)}\right), \quad k=1,\ldots,N$$

加上 $D/h^2$ 和 $\Sigma_a$：

$$\alpha_k = \frac{4D}{h^2}\sin^2\left(\frac{k\pi}{2(N+1)}\right) + \Sigma_a$$

这就是 debug 的核心工具 — 如果代码构造的矩阵对角化后，特征值和解析公式不一致，说明矩阵构造有 bug。


In [ ]:
# ===== 解析特征值验证（小网格，可以手算）=====
N_small = 6
h_small = L / N_small
coeff_small = D / (h_small * h_small)

L_small_mat = diags([-np.ones(N_small-1), 2*np.ones(N_small), -np.ones(N_small-1)], [-1, 0, 1])
A_small = coeff_small * L_small_mat + Sa * diags([np.ones(N_small)], [0])

# 数值特征值
lambda_num = np.sort(np.linalg.eigvalsh(A_small.toarray()))

# 解析特征值
k = np.arange(1, N_small + 1)
lambda_analytic = coeff_small * 4 * np.sin(k * np.pi / (2 * (N_small + 1)))**2 + Sa

error = np.abs(lambda_num - lambda_analytic)
print(f'最大误差: {np.max(error):.2e}')
print(f'{"OK 矩阵构造正确" if np.max(error) < 1e-12 else "FAIL 有 bug!"}')
print(f'\n前 3 个特征值对比:')
print(f'  {"解析":>14s}  {"数值":>14s}  {"误差":>14s}')
for i in range(3):
    print(f'  {lambda_analytic[i]:>14.6f}  {lambda_num[i]:>14.6f}  {error[i]:>14.2e}')


## 4. 双群扩散：快中子 vs 热中子

单群把所有中子当同一能量——但实际物理中，快中子 (MeV) 和热中子 (0.025 eV) 的行为完全不同：

| | 快中子 (群 1) | 热中子 (群 2) |
|---|---|---|
| 扩散能力 | 强 (D1=1.2 cm) | 弱 (D2=0.4 cm) |
| 被吸收概率 | 低 (Sa1=0.008) | 高 (Sa2=0.08) |
| 引起裂变概率 | 低 (nuSf1=0.003) | **高 (nuSf2=0.105)** |
| 主要去向 | 散射慢化→群 2 | 裂变→产生快中子 |

这构成一个**闭环耦合**：快群 → 散射 → 热群 → 裂变 → 快群 → ...

双群矩阵结构（2N x 2N 分块矩阵）：

$$\begin{bmatrix} A_{11} & 0 \\ A_{21} & A_{22} \end{bmatrix} \begin{bmatrix} \phi_1 \\ \phi_2 \end{bmatrix} = \frac{1}{k} \begin{bmatrix} F_{11} & F_{12} \\ 0 & 0 \end{bmatrix} \begin{bmatrix} \phi_1 \\ \phi_2 \end{bmatrix}$$


In [ ]:
# ===== 双群扩散求解 =====
from solver import solve_two_group

result = solve_two_group(L=200, N=150)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(result['x'], result['phi1'], '#e74c3c', linewidth=2, label='Fast (Group 1)')
ax1.plot(result['x'], result['phi2'], '#3498db', linewidth=2, label='Thermal (Group 2)')
ax1.fill_between(result['x'], 0, result['phi1'], color='#e74c3c', alpha=0.08)
ax1.fill_between(result['x'], 0, result['phi2'], color='#3498db', alpha=0.08)
ax1.set_xlabel('Position (cm)')
ax1.set_ylabel('Neutron Flux (normalized)')
ax1.set_title(f'Two-Group Flux Distribution  |  k_eff = {result["k_eff"]:.6f}')
ax1.legend()
ax1.grid(True, alpha=0.25)

ratio = result['phi2'] / (result['phi1'] + 1e-10)
ax2.plot(result['x'], ratio, '#2ecc71', linewidth=2)
ax2.fill_between(result['x'], 0, ratio, color='#2ecc71', alpha=0.08)
ax2.set_xlabel('Position (cm)')
ax2.set_ylabel('Thermal / Fast Ratio')
ax2.set_title('Thermalization phi2/phi1')
ax2.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


## 5. 临界尺寸扫描：堆芯多大才临界？

中子从诞生到被吸收，平均走 sqrt(M^2) ~ 7 cm。如果堆芯太小，中子还没裂变就跑出去了 → 泄漏太大 → 链式反应无法维持。

临界条件（考研必考的公式）：

$$k_{\text{eff}} = \frac{k_\infty}{1 + M^2 B^2} = 1$$

其中 $B^2 = (\pi/L)^2$ 是几何曲率。反解临界尺寸：

$$L_{\text{crit}} = \pi \sqrt{\frac{M^2}{k_\infty - 1}}$$


In [ ]:
# ===== 临界尺寸扫描 =====
from solver import scan_critical_size

cs = scan_critical_size(L_min=40, L_max=400, n_points=30, N=100)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(cs['L_vals'], cs['k_vals'], 'b.-', linewidth=2, markersize=8, label='FDM Numerical')
ax.plot(cs['L_vals'], cs['analytic']['k_buckling'], 'r--', linewidth=2, label='Buckling Approx')
ax.axhline(y=1.0, color='k', linestyle=':', linewidth=1, label='k=1')
ax.axvline(x=cs['L_crit'], color='gray', linestyle=':', linewidth=1,
           label=f'L_crit = {cs["L_crit"]:.0f} cm')
ax.set_xlabel('Slab Thickness L (cm)')
ax.set_ylabel('k_eff')
ax.set_title('Critical Size Scan')
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

print(f'k_inf = {cs["analytic"]["k_inf"]:.4f}')
print(f'M^2 = {cs["analytic"]["M2"]:.1f} cm^2')
print(f'L_crit (FDM) = {cs["L_crit"]:.1f} cm,  k(L_crit) = {cs["k_crit"]:.6f}')


## 6. 二维扩散：从 3 点模板到 5 点模板

一维每个方程只有 3 个系数 → 三对角矩阵。二维每个点 (i,j) 依赖 4 个邻居 → 5 点模板 → 每行 5 个非零元。

**Kronecker 积构造法**（优雅地处理 Nx*Ny 个点的连接关系）：

$$L_{2D} = I_y \otimes L_x + L_y \otimes I_x$$

两行代码，自动把 3600x3600 矩阵的所有拓扑关系算对。


In [ ]:
# ===== 二维双群扩散 =====
from solver_2d import solve_two_group_2d

result_2d = solve_two_group_2d(Lx=160, Ly=160, Nx=40, Ny=40)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

im1 = axes[0].contourf(result_2d['X'], result_2d['Y'], result_2d['phi1'], levels=20, cmap='Reds')
axes[0].set_title('Fast Flux (Group 1)')
axes[0].set_aspect('equal')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

im2 = axes[1].contourf(result_2d['X'], result_2d['Y'], result_2d['phi2'], levels=20, cmap='Blues')
axes[1].set_title('Thermal Flux (Group 2)')
axes[1].set_aspect('equal')
plt.colorbar(im2, ax=axes[1], shrink=0.8)

ratio_2d = result_2d['phi2'] / (result_2d['phi1'] + 1e-12)
im3 = axes[2].contourf(result_2d['X'], result_2d['Y'], ratio_2d, levels=20, cmap='RdYlBu_r')
axes[2].set_title('Thermal / Fast Ratio')
axes[2].set_aspect('equal')
plt.colorbar(im3, ax=axes[2], shrink=0.8)

plt.suptitle(f'2D Two-Group Diffusion  |  k_eff = {result_2d["k_eff"]:.6f}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 7. 点堆动力学：从"照片"到"视频"

前面做的全是**稳态**（画照片）。点堆方程加上了**时间维度**（拍视频）：

$$\frac{dP}{dt} = \frac{\rho(t) - \beta}{\Lambda} P(t) + \sum_{i=1}^6 \lambda_i C_i(t)$$

$$\frac{dC_i}{dt} = \frac{\beta_i}{\Lambda} P(t) - \lambda_i C_i(t)$$

**缓发中子**让反应堆可控：如果没有缓发中子（beta=0），中子代时间 2e-5 s 会让任何正反应性导致指数爆炸。0.7% 的缓发中子将有效代时间延长到 ~0.1s，给了控制手段足够的响应时间。


In [ ]:
# ===== 点堆动力学瞬态演示 =====
from point_kinetics import (
    solve_point_kinetics, KEEPIN_U235,
    reactivity_step, reactivity_rod_ejection,
    prompt_jump, asymptotic_period,
)

beta = KEEPIN_U235['beta']

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# ---- 停堆 ----
r_shutdown = solve_point_kinetics(
    lambda t: reactivity_step(t, -0.005, t_insert=0.0),
    t_span=(0, 100),
)
axes[0, 0].plot(r_shutdown.t, r_shutdown.P, 'b-', linewidth=2)
axes[0, 0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
Pj = prompt_jump(1.0, -0.005)
axes[0, 0].axhline(y=Pj, color='orange', linestyle=':', alpha=0.7, label=f'Prompt Jump={Pj:.2f}')
axes[0, 0].set_title('SCRAM rho=-500 pcm')
axes[0, 0].set_ylabel('P/P0')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.25)

axes[0, 1].plot(r_shutdown.t, r_shutdown.rho * 1e5, 'r-', linewidth=2)
axes[0, 1].set_title('Reactivity (pcm)')
axes[0, 1].grid(True, alpha=0.25)

# ---- Rod Ejection ----
r_eject = solve_point_kinetics(
    lambda t: reactivity_rod_ejection(t, 0.010, t_eject=0.0, tau=0.03),
    t_span=(0, 0.4), max_step=0.001, use_log=True, P_max=1e4,
)
axes[1, 0].plot(r_eject.t, r_eject.P, 'r-', linewidth=2)
axes[1, 0].set_yscale('log')
axes[1, 0].set_title('Rod Ejection rho=+1000 pcm (>beta)')
axes[1, 0].set_ylabel('P/P0')
axes[1, 0].grid(True, alpha=0.25)

axes[1, 1].plot(r_eject.t, r_eject.rho * 1e5, 'r-', linewidth=2)
axes[1, 1].axhline(y=beta*1e5, color='orange', linestyle=':', label=f'beta = {beta*1e5:.0f} pcm')
axes[1, 1].set_title('Reactivity (pcm)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.25)

plt.suptitle('Point Kinetics: Controllable vs Uncontrollable', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'SCRAM prompt jump: P = {Pj:.3f} P0')
print(f'SCRAM asymptotic period: T = {asymptotic_period(-0.005):.0f} s')
print(f'Rod ejection asymptotic period: T = {asymptotic_period(0.010):.3f} s')


## 8. 总结：你刚刚跑通了什么

| 步骤 | 内容 | 考研对接 |
|------|------|----------|
| 1-2 | 扩散方程 -> FDM 离散 -> 幂迭代 | 865 第3章：扩散方程、k_eff 定义 |
| 3 | 解析特征值验证 | Debug 方法论（复试加分项） |
| 4 | 双群扩散（快/热分离） | 865 第4章：多群扩散理论 |
| 5 | 临界尺寸扫描 | 865 重点：临界条件、buckling 公式 |
| 6 | 二维扩散 + Kronecker 积 | 反应堆计算方法 |
| 7 | 点堆动力学（时变） | 865 第7章：反应堆动力学 |

从这个 Notebook 出发，你可以：
- 修改截面数据，模拟不同堆型（快堆 vs 热堆）
- 自己实现 Thomas 算法替代 spsolve
- 用 Streamlit 版本做交互式演示 (`streamlit run app.py`)
- 阅读 NOTES.md 获取完整的数学推导
